# ⚡ MiniMax-H3 4-Step FlashGen LoRA — Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imLeGEnDco55/Minimax4stepsGradio/blob/main/minimax_h3_flashgen_colab.ipynb)

Este notebook ejecuta la aplicación Gradio de **MiniMax-H3 con FlashGen 4-step distillation LoRA** directamente en Google Colab desde el repositorio [`imLeGEnDco55/Minimax4stepsGradio`](https://github.com/imLeGEnDco55/Minimax4stepsGradio), exponiendo la interfaz a través de un túnel seguro de **Cloudflare (`trycloudflare.com`)**.

---

### ⚠️ Requisitos Críticos de Hardware y Acceso

1. **GPU de Alto Rendimiento (A100 80GB Obligatoria)**:
   - El transformer de MiniMax-H3 tiene **33B parámetros** (~66 GB en memoria bfloat16).
   - Requiere una instancia **A100 (80GB)** en Google Colab Pro / Pro+ con **High-RAM**.
   - *No funcionará en GPUs T4 (16GB) ni L4 (24GB) debido a falta de VRAM (OOM).*

2. **Hugging Face Token (Modelo Gated)**:
   - `MiniMaxAI/MiniMax-H3` tiene términos de uso restringidos.
   - Debes haber aceptado la licencia en: [huggingface.co/MiniMaxAI/MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3)
   - Necesitarás un **HF Access Token** con permisos de lectura.

3. **Arquitectura Split**:
   - El text encoder (Qwen3-VL de 62 GB) se delega vía API a `multimodalart/qwen3vl-conditioner`, reduciendo el consumo local y acelerando la generación.

---

In [ ]:
# @title 1. 🔍 Verificar GPU y Recursos del Sistema
import torch
import os

print("=== COMPROBACIÓN DE HARDWARE ===")
if not torch.cuda.is_available():
    print("❌ ERROR: No se ha detectado GPU CUDA.")
    print("👉 Ve al menú superior: 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución'")
    print("👉 Selecciona 'A100 GPU' y activa 'Forma de proceso de alta memoria (High-RAM)'.")
else:
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f"✅ GPU Detectada: {device_name}")
    print(f"✅ VRAM Total: {vram_gb:.2f} GB")
    
    if vram_gb < 75:
        print("\n⚠️ ADVERTENCIA CRÍTICA:")
        print(f"La GPU actual ({device_name} con {vram_gb:.1f} GB) tiene menos de 80 GB de VRAM.")
        print("MiniMax-H3 Transformer requiere ~66 GB en bfloat16. Puede producirse Out Of Memory (OOM).")
    else:
        print("\n🚀 VRAM suficiente detectada para albergar el modelo de 33B parámetros.")

!nvidia-smi


In [ ]:
# @title 2. 📦 Clonar Repositorio e Instalar Dependencias del Sistema
import os

# Instalar ffmpeg para decodificación y ensamblado de video/audio
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Clonar repositorio GitHub personal de imLeGEnDco
REPO_URL = "https://github.com/imLeGEnDco55/Minimax4stepsGradio"
REPO_DIR = "/content/Minimax4stepsGradio"
if not os.path.exists(REPO_DIR):
    print(f"Clonando {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repositorio ya existente en {REPO_DIR}. Actualizando con git pull...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}


In [ ]:
# @title 3. ⚙️ Instalar Dependencias de Python
# Instala PyTorch, Diffusers (PR 14371 con refactor H3), Gradio y utilidades
%cd /content/Minimax4stepsGradio
!pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
# @title 4. 🔑 Autenticación con Hugging Face (Requerido para MiniMax-H3)
from huggingface_hub import login
import os
import getpass

hf_token = None
# 1. Intentar leer desde Colab Secrets si existe
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. Si no, verificar variable de entorno
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

# 3. Si no existe, solicitar interactivamente
if not hf_token:
    print("Ingresa tu Access Token de Hugging Face (Read):")
    hf_token = getpass.getpass("HF Token: ")

if hf_token and hf_token.strip():
    token_clean = hf_token.strip()
    login(token=token_clean)
    os.environ["HF_TOKEN"] = token_clean
    print("✅ Autenticado exitosamente en Hugging Face.")
else:
    print("⚠️ No se ingresó token. Ten en cuenta que la descarga del checkpoint fallará si el repositorio es privado/gated.")


In [ ]:
# @title 5. 🌐 Instalar Cloudflare Tunnel (`cloudflared`)
import os

if not os.path.exists("/usr/local/bin/cloudflared") and not os.path.exists("/usr/bin/cloudflared"):
    print("Descargando binario oficial de cloudflared...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    !rm -f cloudflared-linux-amd64.deb
    print("✅ Cloudflare Tunnel instalado correctamente.")
else:
    print("✅ Cloudflare Tunnel ya se encuentra instalado.")


In [ ]:
# @title 6. 🚀 Lanzar Servidor Gradio con Túnel Cloudflare
import subprocess
import time
import re
import threading
import os

%cd /content/Minimax4stepsGradio

PORT = 7860

# Opcional: habilitar también enlace de respaldo de Gradio si se desea
ENABLE_GRADIO_SHARE = False  # Cambiar a True si deseas además un enlace .gradio.live
if ENABLE_GRADIO_SHARE:
    os.environ["GRADIO_SHARE"] = "True"

print("🌐 Levantando túnel Cloudflare en el puerto", PORT, "...")
cf_cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"]
cf_process = subprocess.Popen(cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_url = None
stop_event = threading.Event()

def monitor_tunnel(proc):
    global public_url
    url_regex = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    for line in proc.stderr:
        found = url_regex.search(line)
        if found:
            public_url = found.group(0)
            print("\n" + "="*65)
            print(f"✨ TU APLICACIÓN ESTÁ DISPONIBLE EN VIVO:")
            print(f"👉 URL Cloudflare: {public_url}")
            print("="*65 + "\n")
            break
        if stop_event.is_set():
            break

thread = threading.Thread(target=monitor_tunnel, args=(cf_process,), daemon=True)
thread.start()

# Esperar brevemente a que el túnel genere la URL pública
for _ in range(25):
    if public_url:
        break
    time.sleep(0.4)

print("⚡ Cargando modelos y arrancando Gradio (esto puede tardar 2-5 minutos en la primera ejecución)...\n")
try:
    !python app.py
except KeyboardInterrupt:
    print("\nDeteniendo aplicación...")
finally:
    stop_event.set()
    cf_process.terminate()
    print("✅ Túnel y servidor cerrados correctamente.")
